## Indexing Showcase for BM25 and Unigram LM

This notebook presents the indexing used for both BM25 and Unigram LM. The combined indexing shown here for both models is in this notebook solely for presentation purposes and is not used in our final search engine. 

It showcases:
- how each collection statistic is calculated
- the calculation done for each term

In [ ]:
# import necessary libraries
import sys
sys.path.append('../')
from collections import defaultdict, Counter
import pandas as pd
import math
from src.PreProcessingPipeline import BM25_PreProcess


In [ ]:
# Load the dataset
df = pd.read_csv("../data/archive-5/archive (2)/bbc-news-data.csv",sep="\t")

df['document'] = df['title'] + " " + df ['content']

In [ ]:
# Preprocess the corpus
corpus = df['document'].to_list()
bm_prepro = BM25_PreProcess(corpus=corpus, set_stemming=True, set_lemmatization=False, set_stopwords=True)
processed_corpus = bm_prepro.get_corpus()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\charl\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Compute collection statistics
N = len(processed_corpus)
doc_lens = [len(article) for article in processed_corpus]
avgdl = (1 / N) * sum(doc_lens)

doc_lens[:10]

def compute_df(term):
    nt = 0
    for article in processed_corpus:
        if term in article:
            nt += 1
    return nt

def compute_idf(term):
    nt = compute_df(term)
    return math.log((N - nt + 0.5) / (nt + 0.5))

def compute_tf(term, doc):
    return doc.count(term)

collection_len = sum(len(doc) for doc in processed_corpus)

def collection_prob(term):
    freq = 0
    for doc in processed_corpus:
        freq += doc.count(term)
    return freq / collection_len

In [ ]:
# Build collection vocabulary statistics
vocab_counter = Counter()
for doc in processed_corpus:
    vocab_counter.update(doc)

# Overall collection statistics
collection_stats = pd.DataFrame([{
    "num_documents": N,
    "total_terms": collection_len,
    "vocabulary_size": len(vocab_counter),
    "avg_doc_length": round(avgdl, 2),
    "min_doc_length": min(doc_lens),
    "max_doc_length": max(doc_lens)
}])
display(collection_stats)

# Sample indexing statistics for selected terms
sample_terms = ["market", "election", "game", "technology"]
rows = []
for term in sample_terms:
    rows.append({
        "term": term,
        "doc_freq": compute_df(term),
        "collection_freq": vocab_counter[term],
        "idf (BM25)": round(compute_idf(term), 4),
        "collection_prob (ULM)": round(collection_prob(term), 6)
    })
display(pd.DataFrame(rows))

,num_documents,total_terms,vocabulary_size,avg_doc_length,min_doc_length,max_doc_length
0,2225,495874,24152,222.86,49,2237


,term,doc_freq,collection_freq,idf (BM25),collection_prob (ULM)
0,market,433,908,1.4195,0.001831
1,election,6,7,5.8332,0.000014
2,game,455,1640,1.3576,0.003307
3,technology,0,0,8.4009,0.000000
